Hälsostudie – Dataanalys
Syftet med analysen är att undersöka deltagarnas hälsodata, inklusive ålder, kön, vikt, längd, blodtryck, kolesterol, rökvanor och sjukdomsförekomst.

Analysen omfattar:

Beskrivande statistik
Datavisualisering
Sjukdomssimulering
Konfidensintervall
Hypotesprövning
Power-simulering
Målet är att identifiera mönster och undersöka skillnader mellan olika grupper med hjälp av statistiska metoder.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import sys
from pathlib import Path

sns.set_theme(style="whitegrid")

project_root = Path.cwd().parent
sys.path.append(f"{project_root}\\src")

# Load dataset
df = pd.read_csv(f"{project_root}/data/health_study_dataset.csv")

# Remove duplicates
df = df.drop_duplicates()

# Convert numerical columns
numeric_columns = [
    "age", "weight", "height",
    "systolic_bp", "cholesterol", "disease"
]

df[numeric_columns] = df[numeric_columns].apply(
    pd.to_numeric, errors="coerce"
)

# Clean categorical columns
df["sex"] = df["sex"].astype("string").str.strip().str.upper()
df["smoker"] = df["smoker"].astype("string").str.strip().str.capitalize()

# Remove missing values
df = df.dropna(
    subset=numeric_columns + ["sex", "smoker"]
).copy()

# Explore dataset
print("Dataset shape:", df.shape)
print("\nMissing values:\n", df.isnull().sum())

display(df.head())
display(df.describe().round(2))

In [ ]:
# Descriptive statistics
columns = [
    "age", "weight", "height",
    "systolic_bp", "cholesterol"
]

statistics = pd.DataFrame({
    "Mean": df[columns].mean(),
    "Median": df[columns].median(),
    "Minimum": df[columns].min(),
    "Maximum": df[columns].max()
})

display(statistics.round(2))

# Create three graphs
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Blood pressure histogram
sns.histplot(
    data=df, x="systolic_bp",
    bins=20, kde=True, ax=axes[0],
    color="steelblue"
)
axes[0].set_title("Blood Pressure Distribution")
axes[0].set_xlabel("Systolic BP (mmHg)")

# 2. Weight boxplot by sex
sns.boxplot(
    data=df, x="sex", y="weight",
    hue="sex", legend=False, ax=axes[1],
    palette="Set2"
)
axes[1].set_title("Weight by Sex")
axes[1].set_ylabel("Weight (kg)")

# 3. Percentage of smokers
smoker_percentage = df["smoker"].value_counts(normalize=True) * 100

axes[2].bar(
    smoker_percentage.index,
    smoker_percentage.values,
    color=["steelblue", "orange"]
)
axes[2].set_title("Percentage of Smokers")
axes[2].set_ylabel("Percentage (%)")

plt.tight_layout()
plt.show()

In [ ]:
# Actual disease proportion
actual_proportion = df["disease"].mean()

print("Actual disease proportion:",
      round(actual_proportion * 100, 2), "%")

# Simulate 1000 people
np.random.seed(42)

simulated_people = np.random.binomial(
    n=1,
    p=actual_proportion,
    size=1000
)

simulated_proportion = simulated_people.mean()

print("Simulated disease proportion:",
      round(simulated_proportion * 100, 2), "%")

print("Difference:",
      round(abs(actual_proportion - simulated_proportion) * 100, 2),
      "percentage points")

# Compare actual and simulated results
plt.figure(figsize=(7, 4))

plt.bar(
    ["Actual", "Simulated"],
    [actual_proportion * 100, simulated_proportion * 100],
    color=["steelblue", "orange"]
)

plt.title("Actual vs Simulated Disease Proportion")
plt.ylabel("Percentage (%)")
plt.show()

In [ ]:
bp = df["systolic_bp"]

n = len(bp)
mean_bp = bp.mean()
std_bp = bp.std(ddof=1)

# Method 1: Normal approximation
standard_error = std_bp / np.sqrt(n)
z_value = stats.norm.ppf(0.975)

lower_normal = mean_bp - z_value * standard_error
upper_normal = mean_bp + z_value * standard_error

# Method 2: Bootstrap
np.random.seed(42)

bootstrap_means = []

for i in range(10000):
    sample = np.random.choice(
        bp, size=n, replace=True
    )
    bootstrap_means.append(sample.mean())

lower_bootstrap = np.percentile(bootstrap_means, 2.5)
upper_bootstrap = np.percentile(bootstrap_means, 97.5)

# Compare confidence intervals
ci_results = pd.DataFrame({
    "Method": ["Normal approximation", "Bootstrap"],
    "Lower limit": [lower_normal, lower_bootstrap],
    "Upper limit": [upper_normal, upper_bootstrap]
})

ci_results["Interval width"] = (
    ci_results["Upper limit"] - ci_results["Lower limit"]
)

print("Mean systolic BP:", round(mean_bp, 2))
display(ci_results.round(2))

In [ ]:
# Divide participants into groups
smokers = df.loc[df["smoker"] == "Yes", "systolic_bp"]
non_smokers = df.loc[df["smoker"] == "No", "systolic_bp"]

# Calculate group means
mean_smokers = smokers.mean()
mean_non_smokers = non_smokers.mean()

print("Mean BP - Smokers:", round(mean_smokers, 2))
print("Mean BP - Non-smokers:", round(mean_non_smokers, 2))

# Welch's t-test
t_statistic, p_two_sided = stats.ttest_ind(
    smokers,
    non_smokers,
    equal_var=False
)

# One-sided test: smokers have higher BP
if mean_smokers > mean_non_smokers:
    p_value = p_two_sided / 2
else:
    p_value = 1 - p_two_sided / 2

alpha = 0.05

print("\nT-statistic:", round(t_statistic, 4))
print("One-sided p-value:", round(p_value, 4))

if p_value < alpha:
    print("Reject H0: Statistical evidence that smokers have higher BP.")
else:
    print("Cannot reject H0: Insufficient statistical evidence.")

# Visual comparison
plt.figure(figsize=(7, 4))

sns.boxplot(
    data=df, x="smoker", y="systolic_bp",
    hue="smoker", legend=False, palette="Set2"
)

plt.title("Blood Pressure: Smokers vs Non-smokers")
plt.xlabel("Smoking status")
plt.ylabel("Systolic BP (mmHg)")
plt.show()

In [ ]:
np.random.seed(42)

number_of_simulations = 5000

# Observed group statistics
n_smokers = len(smokers)
n_non_smokers = len(non_smokers)

mean_s = smokers.mean()
mean_ns = non_smokers.mean()

std_s = smokers.std(ddof=1)
std_ns = non_smokers.std(ddof=1)

significant_results = 0

for i in range(number_of_simulations):

    # Simulate new groups
    simulated_smokers = np.random.normal(
        mean_s, std_s, n_smokers
    )

    simulated_non_smokers = np.random.normal(
        mean_ns, std_ns, n_non_smokers
    )

    # Welch's t-test
    t_stat, p_two_sided = stats.ttest_ind(
        simulated_smokers,
        simulated_non_smokers,
        equal_var=False
    )

    # One-sided p-value
    if simulated_smokers.mean() > simulated_non_smokers.mean():
        p_simulated = p_two_sided / 2
    else:
        p_simulated = 1 - p_two_sided / 2

    if p_simulated < 0.05:
        significant_results += 1

# Calculate power
power = significant_results / number_of_simulations

print("Number of simulations:", number_of_simulations)
print("Significant results:", significant_results)
print("Estimated power:", round(power, 4))
print("Power percentage:", round(power * 100, 2), "%")

# Visualize power
plt.figure(figsize=(6, 4))

plt.bar(
    ["Power", "Not significant"],
    [power * 100, (1 - power) * 100],
    color=["seagreen", "lightcoral"]
)

plt.title("Power Simulation")
plt.ylabel("Percentage (%)")
plt.ylim(0, 100)
plt.show()

# Slutsats och metodval

## 1. Beskrivande analys

Vi undersökte medelvärde, median, minimum och maximum
för ålder, vikt, längd, blodtryck och kolesterol.

Histogrammet visar blodtryckets fördelning.
Boxploten jämför vikt mellan könen.
Stapeldiagrammet visar andelen rökare och icke-rökare.

## 2. Simulering

Vi beräknade den verkliga sjukdomsandelen och simulerade
1000 personer med samma sannolikhet för sjukdom.

Skillnader mellan resultaten beror på slumpmässig variation.

## 3. Konfidensintervall

Vi använde två metoder:

- Normalapproximation använder medelvärde och standardfel.
- Bootstrap använder upprepade stickprov med återläggning.

Vi jämförde intervallens gränser och bredd för att undersöka
om metoderna gav liknande resultat.

## 4. Hypotesprövning

H0: Rökare har inte högre genomsnittligt blodtryck
än icke-rökare.

H1: Rökare har högre genomsnittligt blodtryck
än icke-rökare.

Vi använde Welch's t-test eftersom det jämför två
oberoende gruppers medelvärden utan krav på lika varianser.

Om p-värdet är mindre än 0,05 förkastar vi nollhypotesen.
Annars kan vi inte förkasta den.

Ett statistiskt samband bevisar inte orsakssamband.

## 5. Power-simulering

Vi genomförde 5000 simuleringar för att uppskatta
hur ofta testet hittar den antagna skillnaden.

En hög power innebär större möjlighet att upptäcka
en verklig skillnad.

Simuleringen bygger på observerade gruppvärden och
antagandet om normalfördelning. Resultatet är därför
en uppskattning av testets power.

## 6. Motivering av metodval

- Pandas används för databehandling och statistik.
- NumPy används för simulering och bootstrap.
- Matplotlib och Seaborn används för visualisering.
- Normalapproximation används för ett enkelt
  konfidensintervall för medelvärdet.
- Bootstrap används för att uppskatta osäkerhet
  genom omsampling.
- Welch's t-test används för att jämföra två oberoende
  grupper med potentiellt olika varianser.
- Power-simulering används för att uppskatta testets
  förmåga att upptäcka den antagna skillnaden.

## 7. Källor

1. Pandas:
https://pandas.pydata.org/docs/

2. NumPy:
https://numpy.org/doc/stable/reference/random/

3. SciPy:
https://docs.scipy.org/doc/scipy/reference/stats.html

4. Welch's t-test:
https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_ind.html

5. Seaborn:
https://seaborn.pydata.org/

6. Matplotlib:
https://matplotlib.org/stable/

In [ ]:
import sys

!{sys.executable} -m pip install seaborn